# Natural Language Inference

## Evaluating an ESIM-lite BiLSTM Attention Model

This notebook evaluates the trained Natural Language Inference (NLI) model on the development dataset. The same tokenizer and configuration used during training are loaded to ensure consistent preprocessing. The notebook reports evaluation metrics such as accuracy, precision, recall, and F1 score, and generates predictions for the trial dataset.

## Path

This section defines the file paths for the development dataset, trial dataset, and output predictions. These paths are used to load the trained model and datasets during evaluation.

In [ ]:
import os

# Path to the development dataset (used for evaluation)
DEV_CSV   = "./data/dev.csv"

# Path to the trial/test dataset (used to generate predictions)
TRIAL_CSV = "./data/NLI_trial.csv"

# Output directory for prediction files
OUT_DIR = ""

# File where predictions will be saved
PRED_OUT = os.path.join(OUT_DIR, "predictions.csv")

print("DEV_CSV:", DEV_CSV)
print("TRIAL_CSV:", TRIAL_CSV)
print("OUT_DIR:", OUT_DIR)
print("PRED_OUT:", PRED_OUT)

## Imports

The required libraries are imported for data processing, loading the trained model, tokenising text, and computing evaluation metrics

In [ ]:
# Standard libraries for data handling and model loading
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

# Evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# Used to pad token sequences to fixed length
from tensorflow.keras.preprocessing.sequence import pad_sequences

## Load Configuration

The configuration file saved during training is loaded to ensure that the same sequence length and preprocessing settings are used during evaluation.

In [ ]:
# Load training configuration saved during training
cfg_path = os.path.join(OUT_DIR, "config.json")
cfg = json.load(open(cfg_path))

# Use the same sequence length as used during training
MAX_LEN = int(cfg["MAX_LEN"])

print("Loaded config:", cfg_path)
print("MAX_LEN:", MAX_LEN)

## Load Tokenizer

The tokenizer created during training is loaded so that the same vocabulary mapping is used when converting text into sequences.

In [ ]:
# Load the tokenizer used during training
tok_path = os.path.join(OUT_DIR, "tokenizer.pkl")

with open(tok_path, "rb") as f:
    tokenizer = pickle.load(f)

print("Loaded tokenizer:", tok_path)

## Tokenisation Function

This function converts text into sequences of token IDs using the trained tokenizer and pads them to the fixed maximum sequence length.

In [ ]:
# Convert text to token sequences and pad to MAX_LEN
def tok_pad(series):

    # Convert words to integer token IDs
    seqs = tokenizer.texts_to_sequences(series.astype(str).tolist())

    # Pad sequences so they all have the same length
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")

## Load Model

The best model saved during training is loaded and compiled so it can be used for evaluation and prediction.

In [ ]:
# Path to the best model checkpoint saved during training
model_path = os.path.join(OUT_DIR, "best_model.keras")

# Load the trained model
model = tf.keras.models.load_model(model_path, compile=False)

# Compile the model so evaluation metrics can be computed
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Loaded model:", model_path)

## Evaluate on dev set

The model is evaluated on the development dataset to measure performance using accuracy, precision, recall, and F1 score.

In [ ]:
# Load development dataset (contains labels)
dev_df = pd.read_csv(DEV_CSV)

# Convert text into padded sequences
X_dev_p = tok_pad(dev_df["premise"])
X_dev_h = tok_pad(dev_df["hypothesis"])

# True labels
y_dev = dev_df["label"].astype(int).values

# Predict probabilities
dev_probs = model.predict([X_dev_p, X_dev_h], batch_size=256, verbose=0).ravel()

# Convert probabilities to binary predictions using threshold 0.5
dev_pred = (dev_probs >= 0.5).astype(int)

# Compute evaluation metrics
acc = accuracy_score(y_dev, dev_pred)
p   = precision_score(y_dev, dev_pred)
r   = recall_score(y_dev, dev_pred)
f1  = f1_score(y_dev, dev_pred)

# Print metrics
print(f"DEV Accuracy : {acc:.4f}")
print(f"DEV Precision: {p:.4f}")
print(f"DEV Recall   : {r:.4f}")
print(f"DEV F1       : {f1:.4f}\n")

print(classification_report(y_dev, dev_pred, digits=4))

## Generate Predictions

Predictions are generated for the trial dataset, which does not contain labels. These predictions are saved to a CSV file for submission.

In [ ]:
# Load trial dataset (no labels)
trial_df = pd.read_csv(TRIAL_CSV)

# Convert text into padded sequences
X_t_p = tok_pad(trial_df["premise"])
X_t_h = tok_pad(trial_df["hypothesis"])

# Predict probabilities
probs = model.predict([X_t_p, X_t_h], batch_size=256, verbose=0).ravel()

# Convert probabilities to binary predictions
pred = (probs >= 0.5).astype(int)

# Save predictions to CSV
pd.DataFrame({"prediction": pred}).to_csv(PRED_OUT, index=False)

print("Saved predictions to:", PRED_OUT)